# Customer Churn Prediction — ANN

Study notebook: EDA, preprocessing, and training walked through interactively,
using the same `src/` code `app.py` imports — so this notebook and the app
never diverge.

**Dataset:** "Churn Modelling" — 10,000 bank customers, demographic +
transactional features, target `Exited` (1 = churned, 0 = stayed).


In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import DATA_PATH, DROP_COLUMNS, TARGET_COLUMN


## EDA

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


In [ ]:
df[TARGET_COLUMN].value_counts(normalize=True)
# ~20% churn rate — imbalanced enough that accuracy alone isn't the full picture,
# worth knowing precision/recall too even though the resume bullet reports accuracy.


In [ ]:
df.isna().sum().sum()  # 0 — this dataset has no missing values


In [ ]:
df = df.drop(columns=DROP_COLUMNS)  # RowNumber/CustomerId/Surname: identifiers, not predictive
df.describe()


## Preprocessing

`StandardScaler` for numeric features (ANNs train much better on
standardized inputs — gradient descent converges faster and more stably when
features are on comparable scales, rather than e.g. Balance in the tens of
thousands vs. Tenure in single digits). `OneHotEncoder` for the two
categorical columns (Geography, Gender) since a neural network needs numeric
input, and label-encoding them as 0/1/2 would wrongly imply an ordinal
relationship between categories that don't have one.


In [ ]:
from sklearn.model_selection import train_test_split
from src.preprocessing import build_preprocessor

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = build_preprocessor()
X_train_processed = preprocessor.fit_transform(X_train)   # fit on TRAIN ONLY
X_test_processed = preprocessor.transform(X_test)          # transform only, no refit

print(X_train_processed.shape)


## Model

A small multi-layer ANN: 2 hidden Dense(ReLU) layers, sigmoid output for
binary classification. See `src/model.py` for the full reasoning behind each
architecture choice.


In [ ]:
from src.model import build_ann

model = build_ann(input_dim=X_train_processed.shape[1])
model.summary()


## Training with EarlyStopping

`EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)`:
stops training once validation loss hasn't improved for 10 consecutive
epochs, and rolls the model's weights back to whichever epoch had the best
validation loss — not necessarily the last one. This is the regularization
technique the resume bullet refers to: it prevents the network from
continuing to fit noise in the training set after it's stopped generalizing
better to unseen data.


In [ ]:
from tensorflow import keras

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_processed, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=2,
)


In [ ]:
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.axvline(len(history.history['val_loss']) - early_stopping.patience - 1,
            color='red', linestyle='--', label='best epoch (restored)')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend()
plt.title('Training vs validation loss')
plt.show()


## Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_processed, y_test, verbose=0)
print(f'Test accuracy: {test_accuracy*100:.2f}%')
print(f'Test loss: {test_loss:.4f}')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test_processed, verbose=0) > 0.5).astype(int)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


## Saving artifacts

Both the fitted preprocessor and the trained model are saved — never one
without the other, since a model is meaningless without the exact
preprocessing it was trained against.


In [ ]:
import os
from src.config import MODEL_PATH, PREPROCESSOR_PATH
from src.preprocessing import save_preprocessor

os.makedirs('saved_model', exist_ok=True)
model.save(MODEL_PATH)
save_preprocessor(preprocessor, PREPROCESSOR_PATH)
print('saved')


## Where this goes next

`src/train.py` runs this exact sequence as a single script
(`python -m src.train`). `app.py` (Streamlit) loads the saved model +
preprocessor and serves real-time predictions with visual analytics. See
`README.md` for the full architecture and run instructions.
